# Prism Shader

Prism Shader models light passing through a stack of thin conducting sheets, where rays entangle through a shared surface mode, and bakes the result into an iridescent material shader. You get one ZIP with the same material for Blender, Maya, Houdini, Marmoset, OpenGL, DirectX and MaterialX. (The API names it Entanglement Shader.)

## Setup

In [ ]:
import getpass
import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np

from moth_atlas import Atlas

ENGINE = "entanglement-shader-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "entanglement-shader-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
def read_hdr(data):
    """The first channel of a Radiance .hdr lookup table (flat or run-length rows), as a 2-D array: rows are the
    angle of incidence (0 to pi/2), columns the phase (one period)."""
    header, _, body = data.partition(b"\n\n")
    size, _, body = body.partition(b"\n")
    _, h, _, w = size.split()
    h, w, b, i = int(h), int(w), np.frombuffer(body, np.uint8), 0
    out = np.zeros((h, w))
    for y in range(h):
        if b[i] == 2 and b[i + 1] == 2:  # run-length row: each of R, G, B, E in turn
            i += 4
            row = np.zeros((4, w), np.uint8)
            for c in range(4):
                x = 0
                while x < w:
                    n = int(b[i]); i += 1
                    if n > 128:
                        row[c, x:x + n - 128] = b[i]; i += 1; x += n - 128
                    else:
                        row[c, x:x + n] = b[i:i + n]; i += n; x += n
            r, e = row[0], row[3]
        else:
            px = b[i:i + 4 * w].reshape(w, 4); i += 4 * w
            r, e = px[:, 0], px[:, 3]
        out[y] = np.where(e > 0, r * 2.0 ** (e.astype(int) - 136), 0)
    return out


def tables(path):
    """The reflectance and transmittance tables in a shader ZIP."""
    with zipfile.ZipFile(path) as z:
        return read_hdr(z.read("R_lut.hdr")), read_hdr(z.read("T_lut.hdr"))


def sphere(R, T, size=180, thickness=500):
    """A lit sphere in the material, sampled as the shader does: per wavelength, by viewing angle and phase."""
    h, w = R.shape
    y, x = np.mgrid[1:-1:size * 1j, -1:1:size * 1j]
    inside = x**2 + y**2 < 1
    z = np.sqrt(np.clip(1 - x**2 - y**2, 0, 1))
    light = np.array([-0.4, 0.5, 0.75]) / np.linalg.norm([-0.4, 0.5, 0.75])
    diffuse = np.clip(x * light[0] + y * light[1] + z * light[2], 0, None)
    t = np.arccos(np.clip(z, 0, 1)) / (np.pi / 2) * (h - 1)
    y0 = np.floor(t).astype(int); y1 = np.minimum(h - 1, y0 + 1); fy = t - y0

    def sample(lut, s):
        u = (s - np.floor(s)) * w - 0.5
        x0 = np.floor(u).astype(int) % w; x1 = (x0 + 1) % w; fx = u - np.floor(u)
        row = lambda yy: lut[yy, x0] * (1 - fx) + lut[yy, x1] * fx
        return row(y0) * (1 - fy) + row(y1) * fy

    rgb = np.stack([1 - np.exp(-1.5 * (sample(R, s) * (0.3 + 0.7 * diffuse) + 0.2 * sample(T, s)))
                    for s in (-2 * thickness * z / lam for lam in (650, 530, 470))], -1)
    return np.where(inside[..., None], np.clip(rgb, 0, 1), 0)


def spheres(paths, size=2.4):
    """Show {label: shader ZIP} as lit spheres side by side."""
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size + 0.4), squeeze=False)
    for ax, (label, path) in zip(axes[0], paths.items()):
        ax.imshow(sphere(*tables(path)))
        ax.set_title(label, fontsize=9)
        ax.set_axis_off()
    plt.show()

## Try it

The shader is saved beside this notebook as a ZIP. The sphere is a preview drawn from its lookup tables, the way the shaders sample them.

In [ ]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "style": "peaked",
        "layers": 2,
        "incoming_rays": 8,
        "reflectance": 0.2,
        "interaction": 1,
    },
)
path = job.download(to="shader_peaked.zip", quiet=True)[0]
print(*zipfile.ZipFile(path).namelist(), sep="\n")
spheres({"peaked": path})

## Parameters

What to expect from each:

- `style`: which entangling interaction channels are on. `peaked` (reflection and transmission peak at the same angle as layers grow), `frustrated` (competing channels, a non-monotone colour pattern), `3-body` and `constrained` (less light gets through as more arrives, which has no classical counterpart).
- `layers` (1 and up): conducting sheets in the stack.
- `incoming_rays`: rays entering together; at least `layers`. Layers and rays share a 21-qubit budget, and a run over it is refused.
- `reflectance` (0 to 1): how much one sheet reflects head-on; 0 lets everything through, 1 is a mirror.
- `absorption` (0 to 1): the fraction of the maximum absorption at that reflectance.
- `interaction`: strength of the nonlinear terms. 0 turns them off; the sign changes the colours.
- `resolution`: size of the baked lookup tables (default 60); raise it if close-ups band.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Collision model.** Each layer is an atomically thin conducting sheet. When a ray meets it, the light splits three ways: reflected, transmitted, or absorbed into a surface excitation. Each crossing is a quantum gate acting on all three, so a stack of sheets is a circuit.

**One ray vs many.** With one ray, the circuit reproduces classical thin-film optics exactly. With several rays sharing the same surface mode, they become entangled through it and extra terms appear that classical optics can't express; `style` and `interaction` choose which of those terms act.

**Bake.** The engine simulates the circuit once, over every angle of incidence and phase, and writes the reflectance $R$ and transmittance $T$ into two lookup tables (rows: angle from 0 to $\pi/2$; columns: one period of phase). The shaders read those tables per wavelength at render time, with no quantum computation.

The preview sphere shades each point from $R$ and $T$ at its viewing angle for red, green and blue light (650, 530, 470 nm, 500 nm spacing), so the colour shifts from centre to rim.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Style

Same stack, four interaction patterns.

In [ ]:
paths = {}
for style in ("peaked", "frustrated", "3-body", "constrained"):
    print(f"style={style}")
    job = client.jobs.run(engine=ENGINE, params={"style": style}, quiet=True)
    paths[style] = job.download(to=f"shader_{style}.zip", quiet=True)[0]
spheres(paths)

### Interaction

At 0 only reflectance and absorption shape the colour; positive and negative values of the same size give different patterns.

In [ ]:
paths = {}
for interaction in (-1.0, 0.0, 1.0, 2.0):
    print(f"interaction={interaction}")
    job = client.jobs.run(engine=ENGINE, params={"interaction": interaction}, quiet=True)
    paths[f"interaction {interaction}"] = job.download(to=f"shader_interaction_{interaction}.zip", quiet=True)[0]
spheres(paths)

### Layers and rays

More sheets add more interference between reflections; more rays add more entanglement through the shared mode.

In [ ]:
paths = {}
for layers, rays in ((1, 4), (2, 8), (3, 8), (4, 8)):
    print(f"layers={layers} incoming_rays={rays}")
    job = client.jobs.run(engine=ENGINE, params={"layers": layers, "incoming_rays": rays}, quiet=True)
    paths[f"{layers} layers, {rays} rays"] = job.download(to=f"shader_{layers}x{rays}.zip", quiet=True)[0]
spheres(paths)

### Reflectance

From a faint sheen to a near-mirror.

In [ ]:
paths = {}
for reflectance in (0.05, 0.2, 0.5, 0.8):
    print(f"reflectance={reflectance}")
    job = client.jobs.run(engine=ENGINE, params={"reflectance": reflectance}, quiet=True)
    paths[f"reflectance {reflectance}"] = job.download(to=f"shader_reflectance_{reflectance}.zip", quiet=True)[0]
spheres(paths)

## Questions

**How do I use the shader?**

Unzip it and pick your renderer's file: `.osl` for Blender (Cycles), Maya or Houdini, `.frag` for Marmoset, `.glsl` for OpenGL, `.hlsl` for DirectX, `.mtlx` for MaterialX. Each needs the two lookup tables beside it (`.exr`, or `.hdr`), and the header of each shader says how to bind them.

**Is anything quantum at render time?**

No. The quantum simulation runs once, when the shader is baked; rendering reads the tables like any texture.

**Why was my run refused?**

Layers and rays share a 21-qubit budget, and `incoming_rays` must be at least `layers`. Lower one of them.

**Is this running on quantum hardware?**

No, on a simulator. The material is the same every time you bake the same settings.